# 06 - Interpretacion de Modelos

## Pregunta de negocio: Por que el modelo predice lo que predice?

Construir un modelo preciso es solo la mitad del trabajo. Entender **por que** el modelo
toma sus decisiones es igualmente importante para:

- **Confianza:** Validar que el modelo aprende patrones reales, no artefactos
- **Comunicacion:** Explicar resultados a stakeholders no tecnicos
- **Mejora:** Identificar donde el modelo falla y por que
- **Regulacion:** En muchas industrias, los modelos deben ser explicables

### Tecnicas de interpretacion:
1. Permutation Importance
2. Partial Dependence Plots (PDP)
3. Feature Importance (tree-based vs permutation)
4. Analisis de errores por subgrupos
5. Explicaciones individuales

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

## 1. Cargar modelos entrenados y preparar datos

In [ ]:
# Imports principales
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configuracion
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Imports cargados correctamente')

In [ ]:
# Cargar modelos guardados
model_dir = '../../../models/nyc_taxi/'

# Modelo de propina generosa (clasificacion binaria)
tip_artifact = joblib.load(f'{model_dir}high_tip_classifier.joblib')
tip_model = tip_artifact['model']
tip_features = tip_artifact['feature_cols']
print(f'Modelo de propina cargado: {tip_artifact["model_name"]}')
print(f'  Features: {tip_features}')
print(f'  Metricas: {tip_artifact["metrics"]}')

print()

# Modelo de tipo de viaje (clasificacion multiclase)
trip_artifact = joblib.load(f'{model_dir}trip_type_classifier.joblib')
trip_model = trip_artifact['model']
trip_features = trip_artifact['feature_cols']
print(f'Modelo de tipo de viaje cargado: {trip_artifact["model_name"]}')
print(f'  Features: {trip_features}')
print(f'  Clases: {trip_artifact["class_names"]}')

In [ ]:
# Cargar datos frescos para interpretacion
query = """
SELECT
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS pickup_dow,
    EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
    trip_distance,
    passenger_count,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    rate_code,
    CASE WHEN rate_code IN (2, 3) THEN 1 ELSE 0 END AS is_airport,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    CASE
        WHEN trip_distance > 0 AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) > 0
        THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) / 60.0)
        ELSE 0
    END AS avg_speed_mph
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    payment_type = '1'
    AND fare_amount > 2.5
    AND fare_amount < 200
    AND trip_distance > 0
    AND trip_distance < 50
    AND tip_amount >= 0
    AND total_amount > 0
    AND passenger_count > 0
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
ORDER BY RAND()
LIMIT 50000
"""

df = bq.query_to_dataframe(query)

# Crear targets
df['tip_pct'] = (df['tip_amount'] / df['fare_amount']) * 100
df = df[(df['tip_pct'] >= 0) & (df['tip_pct'] <= 100)].copy()
df['high_tip'] = (df['tip_pct'] > 20).astype(int)

# Preparar features para el modelo de propina
X_tip = df[tip_features].fillna(0)
y_tip = df['high_tip']

print(f'Datos cargados: {len(df):,} registros')
print(f'Features modelo propina: {len(tip_features)}')

## 2. Permutation Importance: Modelo de propina

**Permutation Importance** mide cuanto empeora el rendimiento del modelo cuando
se barajan aleatoriamente los valores de una feature. Si barajar una feature
reduce mucho el rendimiento, esa feature es importante.

**Ventajas:**
- Funciona con cualquier modelo (model-agnostic)
- Mide importancia en el conjunto de test (no confunde con overfitting)
- Captura interacciones entre features

**Desventajas:**
- Puede subestimar features correlacionadas

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance para el modelo de propina
perm_tip = permutation_importance(
    tip_model, X_tip, y_tip,
    n_repeats=10,
    random_state=42,
    scoring='f1',
    n_jobs=-1
)

# Ordenar por importancia
perm_tip_df = pd.DataFrame({
    'Feature': tip_features,
    'Importance_mean': perm_tip.importances_mean,
    'Importance_std': perm_tip.importances_std
}).sort_values('Importance_mean', ascending=False)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(perm_tip_df)))
ax.barh(perm_tip_df['Feature'], perm_tip_df['Importance_mean'],
        xerr=perm_tip_df['Importance_std'], color=colors, edgecolor='black')
ax.set_xlabel('Reduccion en F1-Score al permutar', fontsize=12)
ax.set_title('Permutation Importance - Modelo de Propina Generosa',
             fontweight='bold', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Importancia por permutacion (modelo propina):')
print(perm_tip_df.to_string(index=False))

## 3. Permutation Importance: Modelo de tipo de viaje

Comparemos que features importan para clasificar el **tipo de viaje** vs las que
importan para predecir la **propina generosa**. Las diferencias nos dicen que
aspectos del viaje influyen en cada prediccion.

In [ ]:
# Preparar datos para el modelo de tipo de viaje
# Necesitamos crear el target de tipo de viaje
def classify_trip(row):
    hour = row['pickup_hour']
    dow = row['pickup_dow']
    distance = row['trip_distance']
    rate = row['rate_code']
    speed = row['avg_speed_mph']
    
    if rate in [2, 3]:
        return 'airport'
    if hour >= 22 or hour < 5:
        return 'nocturno'
    is_weekday = dow in [2, 3, 4, 5, 6]
    is_rush_hour = (7 <= hour <= 9) or (17 <= hour <= 19)
    if is_weekday and is_rush_hour and 1 <= distance <= 10:
        return 'commute'
    is_weekend = dow in [1, 7]
    if is_weekend and 1 <= distance <= 8 and speed < 15:
        return 'turistico'
    if distance < 1:
        return 'corto'
    if distance < 3:
        return 'corto'
    elif is_weekday:
        return 'commute'
    else:
        return 'turistico'

df['trip_type'] = df.apply(classify_trip, axis=1)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_trip = le.fit_transform(df['trip_type'])

X_trip = df[trip_features].fillna(0)

# Permutation importance para el modelo de tipo de viaje
perm_trip = permutation_importance(
    trip_model, X_trip, y_trip,
    n_repeats=10,
    random_state=42,
    scoring='f1_macro',
    n_jobs=-1
)

perm_trip_df = pd.DataFrame({
    'Feature': trip_features,
    'Importance_mean': perm_trip.importances_mean,
    'Importance_std': perm_trip.importances_std
}).sort_values('Importance_mean', ascending=False)

print('Importancia por permutacion (modelo tipo de viaje):')
print(perm_trip_df.to_string(index=False))

In [ ]:
# Comparacion lado a lado
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Modelo propina
axes[0].barh(perm_tip_df['Feature'], perm_tip_df['Importance_mean'],
             xerr=perm_tip_df['Importance_std'], color='#3498db', edgecolor='black')
axes[0].set_title('Modelo: Propina Generosa', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Reduccion en F1')
axes[0].invert_yaxis()

# Modelo tipo de viaje
axes[1].barh(perm_trip_df['Feature'], perm_trip_df['Importance_mean'],
             xerr=perm_trip_df['Importance_std'], color='#e74c3c', edgecolor='black')
axes[1].set_title('Modelo: Tipo de Viaje', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Reduccion en F1 macro')
axes[1].invert_yaxis()

plt.suptitle('Comparacion de Importancia: Que importa para cada prediccion?',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Observaciones:')
print('- Para propina: las features monetarias (fare, total) tienden a ser mas importantes')
print('- Para tipo de viaje: las features temporales y de distancia dominan')
print('- Esto tiene sentido: el tipo de viaje depende de cuando/donde, la propina de cuanto')

## 4. Partial Dependence Plots (PDP)

Los **PDP** muestran el efecto marginal promedio de una feature sobre la prediccion
del modelo. Nos permiten ver la **relacion funcional** entre una feature y el output.

**Interpretacion:**
- Eje X: valores de la feature
- Eje Y: efecto marginal sobre la prediccion
- Una linea ascendente = la feature tiene efecto positivo al aumentar
- Una linea plana = la feature no afecta la prediccion en ese rango

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Submuestra para que los PDPs se calculen mas rapido
sample_idx = np.random.choice(len(X_tip), size=min(5000, len(X_tip)), replace=False)
X_tip_sample = X_tip.iloc[sample_idx]

# PDP: trip_distance -> probabilidad de propina generosa
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Distancia -> Propina
PartialDependenceDisplay.from_estimator(
    tip_model, X_tip_sample,
    features=['trip_distance'],
    kind='average',
    ax=axes[0]
)
axes[0].set_title('Distancia -> P(Propina Generosa)', fontweight='bold')
axes[0].set_xlabel('Distancia del viaje (millas)')
axes[0].set_ylabel('Efecto parcial')

# 2. Hora -> Propina (esperamos propinas mas altas en la noche)
PartialDependenceDisplay.from_estimator(
    tip_model, X_tip_sample,
    features=['pickup_hour'],
    kind='average',
    ax=axes[1]
)
axes[1].set_title('Hora -> P(Propina Generosa)', fontweight='bold')
axes[1].set_xlabel('Hora del dia')
axes[1].set_ylabel('Efecto parcial')

# 3. Velocidad -> Propina
PartialDependenceDisplay.from_estimator(
    tip_model, X_tip_sample,
    features=['avg_speed_mph'],
    kind='average',
    ax=axes[2]
)
axes[2].set_title('Velocidad Promedio -> P(Propina Generosa)', fontweight='bold')
axes[2].set_xlabel('Velocidad promedio (mph)')
axes[2].set_ylabel('Efecto parcial')

plt.suptitle('Partial Dependence Plots - Modelo de Propina',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Interpretacion de los PDPs:')
print('- Distancia: relacion esperada con la probabilidad de propina generosa')
print('- Hora: las propinas nocturnas tienden a ser mas generosas?')
print('- Velocidad: viajes mas rapidos (menos trafico) pueden correlacionar con mejor propina')

## 5. PDP 2D: Interaccion distancia x hora -> propina

Los PDP bidimensionales muestran como **dos features interactuan** para afectar
la prediccion. El mapa de calor resultante revela combinaciones de valores
que producen predicciones altas o bajas.

In [ ]:
# PDP 2D: distancia x hora -> propina generosa
fig, ax = plt.subplots(figsize=(12, 8))

# Obtener indices de las features
dist_idx = tip_features.index('trip_distance')
hour_idx = tip_features.index('pickup_hour')

display = PartialDependenceDisplay.from_estimator(
    tip_model, X_tip_sample,
    features=[(dist_idx, hour_idx)],
    kind='average',
    ax=ax
)

ax.set_title('PDP 2D: Distancia x Hora -> P(Propina Generosa)',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('Interpretacion del PDP 2D:')
print('- Las zonas mas oscuras indican mayor probabilidad de propina generosa')
print('- Buscamos combinaciones de distancia y hora que maximizan/minimizan la propina')
print('- Las interacciones revelan patrones que los PDPs univariados no capturan')

## 6. Feature Importance: Tree-based vs Permutation

Los modelos basados en arboles tienen una medida de importancia nativa
(basada en la reduccion de impureza de Gini o la ganancia de informacion).
Comparemosla con la importancia por permutacion.

**Importancia basada en arboles (Gini/Gain):**
- Mide cuanto reduce cada feature la impureza en los nodos del arbol
- Rapida de calcular (viene gratis con el entrenamiento)
- **Sesgo:** favorece features con alta cardinalidad y puede ser enganiosa

**Importancia por permutacion:**
- Mide cuanto empeora el modelo al barajar una feature
- Mas confiable, pero mas lenta de calcular
- Funciona en el test set (no confunde con overfitting)

In [ ]:
# Comparar importancias para el modelo de propina
# Solo funciona si el modelo subyacente es tree-based
try:
    # Intentar obtener feature_importances_ (tree-based)
    if hasattr(tip_model, 'feature_importances_'):
        tree_importance = tip_model.feature_importances_
    else:
        print('El modelo no tiene feature_importances_ nativo.')
        print('Usando solo permutation importance.')
        tree_importance = None
except Exception as e:
    print(f'Error obteniendo importancias: {e}')
    tree_importance = None

if tree_importance is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Tree-based importance
    tree_imp_df = pd.DataFrame({
        'Feature': tip_features,
        'Importance': tree_importance
    }).sort_values('Importance', ascending=False)
    
    axes[0].barh(tree_imp_df['Feature'], tree_imp_df['Importance'],
                 color='#2ecc71', edgecolor='black')
    axes[0].set_title('Importancia basada en Arboles (Gini/Gain)',
                      fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Importancia')
    axes[0].invert_yaxis()
    
    # Permutation importance (ya calculada)
    axes[1].barh(perm_tip_df['Feature'], perm_tip_df['Importance_mean'],
                 color='#e74c3c', edgecolor='black')
    axes[1].set_title('Importancia por Permutacion',
                      fontweight='bold', fontsize=13)
    axes[1].set_xlabel('Reduccion en F1')
    axes[1].invert_yaxis()
    
    plt.suptitle('Comparacion de Metodos de Importancia de Features',
                 fontweight='bold', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Correlacion entre ambas medidas
    merged = tree_imp_df.merge(
        perm_tip_df[['Feature', 'Importance_mean']],
        on='Feature'
    )
    corr = merged['Importance'].corr(merged['Importance_mean'])
    print(f'\nCorrelacion entre ambas medidas: {corr:.3f}')
    print('Una correlacion alta indica que ambos metodos coinciden en que features importan.')
    print('Discrepancias pueden indicar features con alta cardinalidad (infladas en Gini)')
    print('o features correlacionadas (subestimadas en permutacion).')
else:
    print('Solo se dispone de permutation importance para este modelo.')
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(perm_tip_df['Feature'], perm_tip_df['Importance_mean'],
            xerr=perm_tip_df['Importance_std'], color='#e74c3c', edgecolor='black')
    ax.set_title('Permutation Importance - Modelo Propina', fontweight='bold')
    ax.set_xlabel('Reduccion en F1')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 7. Analisis de errores por subgrupo

Un modelo puede tener buen rendimiento global pero fallar en subgrupos especificos.
Analizar los errores por segmentos nos ayuda a identificar:
- Sesgos del modelo
- Zonas donde se necesitan mas datos
- Subpoblaciones que requieren modelos especializados

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Predicciones del modelo de propina
y_pred_tip = tip_model.predict(X_tip)
df['pred_high_tip'] = y_pred_tip
df['tip_correct'] = (df['high_tip'] == df['pred_high_tip']).astype(int)
df['tip_error'] = 1 - df['tip_correct']

print(f'Accuracy global: {df["tip_correct"].mean():.4f}')
print(f'Tasa de error global: {df["tip_error"].mean():.4f}')

In [ ]:
# 7a. Analisis de error por hora del dia
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

error_by_hour = df.groupby('pickup_hour').agg(
    tasa_error=('tip_error', 'mean'),
    n_viajes=('tip_error', 'count')
).reset_index()

axes[0].bar(error_by_hour['pickup_hour'], error_by_hour['tasa_error'],
            color='#e74c3c', edgecolor='black', alpha=0.8)
axes[0].axhline(y=df['tip_error'].mean(), color='black', linestyle='--',
                label=f'Error promedio ({df["tip_error"].mean():.3f})')
axes[0].set_xlabel('Hora del dia')
axes[0].set_ylabel('Tasa de error')
axes[0].set_title('Tasa de Error por Hora', fontweight='bold', fontsize=13)
axes[0].legend()
axes[0].set_xticks(range(24))

# 7b. Volumen por hora (contexto)
axes[1].bar(error_by_hour['pickup_hour'], error_by_hour['n_viajes'],
            color='#3498db', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Hora del dia')
axes[1].set_ylabel('Cantidad de viajes')
axes[1].set_title('Volumen de Viajes por Hora', fontweight='bold', fontsize=13)
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.show()

# Horas con mayor error
worst_hours = error_by_hour.nlargest(3, 'tasa_error')
best_hours = error_by_hour.nsmallest(3, 'tasa_error')
print('Horas con MAYOR tasa de error:')
for _, row in worst_hours.iterrows():
    print(f'  Hora {int(row["pickup_hour"]):02d}: {row["tasa_error"]:.3f} ({int(row["n_viajes"]):,} viajes)')
print('\nHoras con MENOR tasa de error:')
for _, row in best_hours.iterrows():
    print(f'  Hora {int(row["pickup_hour"]):02d}: {row["tasa_error"]:.3f} ({int(row["n_viajes"]):,} viajes)')

In [ ]:
# 7c. Analisis de error por rango de tarifa
fare_bins = [0, 5, 10, 15, 20, 30, 50, 100, 200]
fare_labels = ['$0-5', '$5-10', '$10-15', '$15-20', '$20-30', '$30-50', '$50-100', '$100+']
df['fare_range'] = pd.cut(df['fare_amount'], bins=fare_bins, labels=fare_labels)

error_by_fare = df.groupby('fare_range', observed=True).agg(
    tasa_error=('tip_error', 'mean'),
    n_viajes=('tip_error', 'count'),
    pct_high_tip=('high_tip', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(error_by_fare)), error_by_fare['tasa_error'],
              color='#9b59b6', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(error_by_fare)))
ax.set_xticklabels(error_by_fare['fare_range'], rotation=0)
ax.axhline(y=df['tip_error'].mean(), color='black', linestyle='--',
           label=f'Error promedio ({df["tip_error"].mean():.3f})')
ax.set_xlabel('Rango de Tarifa')
ax.set_ylabel('Tasa de Error')
ax.set_title('Tasa de Error por Rango de Tarifa', fontweight='bold', fontsize=14)
ax.legend()

# Anotar cantidad de viajes
for i, (bar, n) in enumerate(zip(bars, error_by_fare['n_viajes'])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'n={n:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nError por rango de tarifa:')
print(error_by_fare.to_string(index=False))
print('\nObservacion: El modelo suele tener mas dificultad con tarifas')
print('extremas (muy bajas o muy altas), donde hay menos datos de entrenamiento.')

In [ ]:
# 7d. Analisis de error por tipo de viaje
error_by_type = df.groupby('trip_type').agg(
    tasa_error=('tip_error', 'mean'),
    n_viajes=('tip_error', 'count'),
    tip_pct_mean=('tip_pct', 'mean')
).sort_values('tasa_error', ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
colors_type = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
ax.bar(error_by_type['trip_type'], error_by_type['tasa_error'],
       color=colors_type[:len(error_by_type)], edgecolor='black')
ax.axhline(y=df['tip_error'].mean(), color='black', linestyle='--',
           label=f'Error promedio ({df["tip_error"].mean():.3f})')
ax.set_xlabel('Tipo de Viaje')
ax.set_ylabel('Tasa de Error')
ax.set_title('Tasa de Error del Modelo de Propina por Tipo de Viaje',
             fontweight='bold', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print('Error por tipo de viaje:')
print(error_by_type.to_string(index=False))

## 8. Explicaciones de predicciones individuales

Seleccionamos 5 viajes representativos y explicamos por que el modelo predice
lo que predice para cada uno. Usamos las contribuciones de features
para entender cada prediccion.

In [ ]:
# Seleccionar 5 viajes representativos
np.random.seed(42)

# Elegir viajes variados
representative_trips = []

# 1. Viaje con propina generosa bien clasificado (True Positive)
tp_mask = (df['high_tip'] == 1) & (df['pred_high_tip'] == 1)
if tp_mask.sum() > 0:
    representative_trips.append(('Verdadero Positivo\n(propina generosa correcta)', df[tp_mask].sample(1).index[0]))

# 2. Viaje sin propina generosa bien clasificado (True Negative)
tn_mask = (df['high_tip'] == 0) & (df['pred_high_tip'] == 0)
if tn_mask.sum() > 0:
    representative_trips.append(('Verdadero Negativo\n(no generosa correcta)', df[tn_mask].sample(1).index[0]))

# 3. Falso Positivo (predijo generosa pero no lo fue)
fp_mask = (df['high_tip'] == 0) & (df['pred_high_tip'] == 1)
if fp_mask.sum() > 0:
    representative_trips.append(('Falso Positivo\n(predijo generosa, no lo fue)', df[fp_mask].sample(1).index[0]))

# 4. Falso Negativo (no predijo generosa pero si lo fue)
fn_mask = (df['high_tip'] == 1) & (df['pred_high_tip'] == 0)
if fn_mask.sum() > 0:
    representative_trips.append(('Falso Negativo\n(no predijo generosa, si lo fue)', df[fn_mask].sample(1).index[0]))

# 5. Viaje al aeropuerto
airport_mask = df['is_airport'] == 1
if airport_mask.sum() > 0:
    representative_trips.append(('Viaje al aeropuerto', df[airport_mask].sample(1).index[0]))

print(f'Viajes representativos seleccionados: {len(representative_trips)}')
for desc, idx in representative_trips:
    row = df.loc[idx]
    print(f'\n--- {desc.replace(chr(10), " ")} ---')
    print(f'  Distancia: {row["trip_distance"]:.1f} mi, Tarifa: ${row["fare_amount"]:.2f}')
    print(f'  Propina: ${row["tip_amount"]:.2f} ({row["tip_pct"]:.1f}%)')
    print(f'  Real: {"Generosa" if row["high_tip"] else "No generosa"}')
    print(f'  Prediccion: {"Generosa" if row["pred_high_tip"] else "No generosa"}')

In [ ]:
# Explicar cada prediccion usando contribuciones de features
# Metodo: comparar la prediccion del viaje con la prediccion media
# y atribuir la diferencia a cada feature

# Prediccion media (baseline)
if hasattr(tip_model, 'predict_proba'):
    baseline_prob = tip_model.predict_proba(X_tip)[:, 1].mean()
else:
    baseline_prob = y_tip.mean()

fig, axes = plt.subplots(len(representative_trips), 1,
                         figsize=(14, 5 * len(representative_trips)))
if len(representative_trips) == 1:
    axes = [axes]

for ax_idx, (desc, trip_idx) in enumerate(representative_trips):
    trip_features_values = X_tip.loc[trip_idx]
    trip_row = df.loc[trip_idx]
    
    if hasattr(tip_model, 'predict_proba'):
        trip_prob = tip_model.predict_proba(trip_features_values.values.reshape(1, -1))[0, 1]
    else:
        trip_prob = tip_model.predict(trip_features_values.values.reshape(1, -1))[0]
    
    # Calcular contribucion aproximada de cada feature por permutacion individual
    contributions = []
    for i, feat in enumerate(tip_features):
        # Crear copia y reemplazar feature con la media
        X_modified = trip_features_values.values.copy().reshape(1, -1)
        X_modified[0, i] = X_tip[feat].mean()  # Reemplazar con media
        
        if hasattr(tip_model, 'predict_proba'):
            modified_prob = tip_model.predict_proba(X_modified)[0, 1]
        else:
            modified_prob = tip_model.predict(X_modified)[0]
        
        contributions.append({
            'Feature': feat,
            'Valor': trip_features_values[feat],
            'Contribucion': trip_prob - modified_prob  # Positivo = empuja hacia propina generosa
        })
    
    contrib_df = pd.DataFrame(contributions).sort_values('Contribucion', key=abs, ascending=True)
    
    colors_contrib = ['#e74c3c' if c < 0 else '#2ecc71' for c in contrib_df['Contribucion']]
    ax = axes[ax_idx]
    
    bars = ax.barh(
        [f"{row['Feature']} = {row['Valor']:.1f}" for _, row in contrib_df.iterrows()],
        contrib_df['Contribucion'],
        color=colors_contrib, edgecolor='black'
    )
    ax.axvline(x=0, color='black', linewidth=0.8)
    
    real = 'Generosa' if trip_row['high_tip'] else 'No generosa'
    pred = 'Generosa' if trip_row['pred_high_tip'] else 'No generosa'
    ax.set_title(
        f'{desc.replace(chr(10), " ")} | P={trip_prob:.3f} | Real: {real} | Pred: {pred}',
        fontweight='bold', fontsize=12
    )
    ax.set_xlabel('Contribucion a P(propina generosa)')

plt.tight_layout()
plt.show()

print('Interpretacion:')
print('- Barras VERDES: la feature empuja hacia predecir propina generosa')
print('- Barras ROJAS: la feature empuja hacia predecir no generosa')
print('- La longitud indica la magnitud del efecto')

## 9. Resumen: Insights accionables de la interpretacion

### Hallazgos clave de la interpretacion de modelos:

#### Modelo de Propina Generosa:
1. **Features monetarias dominan:** La tarifa y el monto total son los predictores
   mas importantes. Esto sugiere que las propinas generosas estan mas relacionadas
   con el costo del viaje que con las circunstancias temporales.

2. **Efecto de la hora:** Las propinas tienden a variar por hora del dia,
   con posibles picos en horarios nocturnos (efecto de socializar/cenar).

3. **Viajes al aeropuerto:** Los viajes con tarifa de aeropuerto tienen
   patrones de propina diferentes al promedio.

#### Analisis de Errores:
4. **Errores por hora:** El modelo tiene rendimiento variable segun la hora,
   sugiriendo que los patrones de propina cambian a lo largo del dia.

5. **Errores por tarifa:** Las tarifas extremas (muy bajas o muy altas)
   son mas dificiles de clasificar, probablemente por menor representacion
   en los datos de entrenamiento.

6. **Errores por tipo de viaje:** Ciertos tipos de viaje (nocturno, turistico)
   pueden tener patrones de propina mas impredecibles.

### Recomendaciones de negocio:
- Los conductores pueden optimizar su servicio enfocandose en los factores
  que mas influyen en las propinas generosas
- Las plataformas pueden usar estos insights para personalizar sugerencias
  de propina en la app
- Para mejorar el modelo: considerar features adicionales como zona de pickup/dropoff,
  condiciones climaticas, y eventos especiales

In [ ]:
# Resumen final con las metricas y features mas importantes
print('='*70)
print('RESUMEN DE INTERPRETACION DE MODELOS')
print('='*70)

print('\n--- Modelo de Propina Generosa ---')
print(f'Tipo: {tip_artifact["model_name"]}')
print(f'Metricas: {tip_artifact["metrics"]}')
print('\nTop 5 features (permutation importance):')
for i, (_, row) in enumerate(perm_tip_df.head(5).iterrows()):
    print(f'  {i+1}. {row["Feature"]}: {row["Importance_mean"]:.4f}')

print(f'\n--- Modelo de Tipo de Viaje ---')
print(f'Tipo: {trip_artifact["model_name"]}')
print(f'F1 Macro: {trip_artifact["f1_macro"]:.4f}')
print('\nTop 5 features (permutation importance):')
for i, (_, row) in enumerate(perm_trip_df.head(5).iterrows()):
    print(f'  {i+1}. {row["Feature"]}: {row["Importance_mean"]:.4f}')

print('\n--- Subgrupos con Mayor Error ---')
print('El modelo de propina tiene dificultades con:')
print('  - Tarifas extremas (muy bajas o muy altas)')
print('  - Ciertas horas del dia')
print('  - Tipos de viaje especificos')
print('='*70)